# 1. Loading Our Dataset: `rag.pdf`

**RAG Pipeline Series — Notebook 1**

Every RAG pipeline starts the same way: get your raw content into a common `Document` object
(page content + metadata) so every later step (chunking, embedding, retrieval) doesn't care
where the data came from.

In this notebook we will:
1. Get `rag.pdf` into the Colab runtime.
2. Load it with LangChain's `PyPDFLoader`, which produces one `Document` per PDF page.
3. Inspect a few sample pages (content + metadata) so we know what we're working with.
4. Build a quick per-page stats table (character / word counts) with pandas.

This notebook only *loads* the data — chunking, embeddings, and retrieval are covered in the
notebooks that follow.

## Setup

In [2]:
%pip install -q -U langchain langchain-community pypdf pandas

Note: you may need to restart the kernel to use updated packages.


## 1. Getting `rag.pdf` into Colab

This notebook expects a file named `rag.pdf` in the Colab runtime.

- **In Colab:** run the upload cell below and pick `rag.pdf` from your machine (it lands at
  `/content/rag.pdf`).
- **Running locally instead:** skip the upload cell — the fallback path below looks for the
  file at `../dataset/rag.pdf` (this repo's dataset folder).

In [ ]:
# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
from rag_utils import maybe_colab_upload

maybe_colab_upload()

In [1]:
from rag_utils import resolve_pdf_path

# Prefer the Colab upload location; fall back to this repo's dataset/ folder
# when running locally (rag-notebooks/ and dataset/ are sibling folders).
PDF_PATH = resolve_pdf_path()
print("Using PDF_PATH:", PDF_PATH)

Using PDF_PATH: ..\dataset\rag.pdf


## 2. Loading the PDF with `PyPDFLoader`

`PyPDFLoader` reads the PDF and returns **one `Document` per page**. Each `Document` gets:
- `page_content` — the extracted text of that page
- `metadata["source"]` — the file path
- `metadata["page"]` — the zero-indexed page number

This page-level granularity is exactly what later notebooks (chunking, embeddings) build on
top of.

In [2]:
from rag_utils import load_pdf

pages = load_pdf(PDF_PATH)  # List[Document], one per PDF page

print(f"Loaded {len(pages)} pages from {PDF_PATH}")

Loaded 58 pages from ..\dataset\rag.pdf


## 3. Inspecting a few sample pages

Let's look at the metadata and a content preview for the first few pages, to sanity-check the
extraction before we build anything on top of it.

In [3]:
# Print metadata + a short content preview for the first 3 pages
for doc in pages[:3]:
    print("metadata:", doc.metadata)
    print("content preview:", doc.page_content[:300].replace("\n", " "), "...")
    print("-" * 80)

metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-13T16:14:53+00:00', 'author': 'AI Paatshal', 'keywords': '', 'moddate': '2026-07-13T16:14:53+00:00', 'subject': 'Retrieval-Augmented Generation', 'title': 'RAG to Agentic RAG — Comprehensive Course', 'trapped': '/False', 'source': '..\\dataset\\rag.pdf', 'total_pages': 58, 'page': 0, 'page_label': '1'}
content preview: RAG to Agentic RAG — Comprehensive Course Page 1  RAG to Agentic RAG  A Comprehensive Course — From Foundations to Production  Agentic Systems 15 In-Depth Chapters  Feynman Explanations Real-World Examples  Production Best  Practices Ch 01 Introduction to RAG Ch 09 Augmentation Ch 02 Evolution of Re ...
--------------------------------------------------------------------------------
metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-13T16:14:53+00:00', 'author': 'AI Paatshal', 'keywords': '', 

## 4. Quick per-page stats

A small pandas table of character / word counts per page — useful for spotting pages that
extracted oddly (e.g. near-empty pages from scanned images or figures).

In [4]:
import pandas as pd

stats = pd.DataFrame([
    {
        "page": doc.metadata.get("page"),
        "char_count": len(doc.page_content),
        "word_count": len(doc.page_content.split()),
    }
    for doc in pages
])

stats.describe()

,page,char_count,word_count
count,58.000000,58.00000,58.000000
mean,28.500000,1944.87931,286.741379
std,16.886879,737.15782,109.442857
min,0.000000,156.00000,25.000000
25%,14.250000,1911.50000,278.750000
50%,28.500000,2218.50000,323.000000
75%,42.750000,2395.00000,357.000000
max,57.000000,2801.00000,412.000000


In [5]:
# Full per-page breakdown
stats

,page,char_count,word_count
0,0,591,96
1,1,2475,374
2,2,2578,379
3,3,2382,372
4,4,1421,227
5,5,2699,397
6,6,2406,338
7,7,2634,399
8,8,156,25
9,9,2347,327


## Takeaways

- `PyPDFLoader` turns `rag.pdf` into one `Document` per page — `page_content` (text) +
  `metadata` (`source`, `page`).
- Always spot-check a few pages and look at per-page char/word counts before trusting an
  extraction — PDFs with scanned pages, tables, or multi-column layouts often extract poorly.
- `pages` (the list of `Document` objects built here) is the raw input the rest of this series
  builds on.

**Next up (notebook 2):** now that we can load `rag.pdf` into `Document` objects, we'll look at
*chunking* — splitting these pages into retrieval-sized pieces.